In [ ]:
import time
import datetime
import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
import os

# 定義 Excel 檔案名稱與工作表名稱
EXCEL_FILE = "factory_production_data.xlsx"
SHEET_NAME = "Live_Data"

print("套件載入成功，準備初始化 Excel 檔案...")

In [ ]:
# ==========================================
# 1. 初始化 Excel 檔案與標頭
# ==========================================
# 定義 Excel 表格的第一列欄位名稱 (標頭)
headers = ["序號", "時間", "晶片長度(mm)", "鍋爐溫度(°C)", "廠房濕度(%)"]

# 檢查 Excel 檔案是否還不存在
if not os.path.exists(EXCEL_FILE):
    wb = Workbook()  # 建立一個全新的 Excel 活頁簿物件
    ws = wb.active  # 取得預設的作用中工作表
    ws.title = SHEET_NAME  # 將工作表重新命名為 "Live_Data"
    ws.append(headers)  # 將標頭列寫入工作表的第一列
    wb.save(EXCEL_FILE)  # 存檔，建立實體 xlsx 檔案
    print(f"已建立新檔案: {EXCEL_FILE}")
else:
    # 檔案已存在，就不需要重建，直接沿用舊檔
    print(f"檔案 {EXCEL_FILE} 已存在，將繼續追加資料。")

In [ ]:
# ==========================================
# 2. 數據生成 logic 與 實時模擬器
# ==========================================
# 這一段的目標：用程式模擬工廠感測器，每呼叫一次就產生一筆假資料

class FactoryDataGenerator:
    # 工廠數據產生器類別：封裝序號、時間、溫度狀態，並提供產生各欄位的方法
    def __init__(self, start_hour=8, start_minute=0):
        # 建構子：建立物件時初始化，預設模擬從今天 08:00 開始
        self.seq = 1  # 生產序號計數器，從 1 開始，每產生一筆 +1
        # 設定假裝的起始時間（預設當天 08:00）
        today = datetime.datetime.now().date()  # 取得今天的日期 (年-月-日)
        self.current_sim_time = datetime.datetime(today.year, today.month, today.day, start_hour, start_minute)  # 組合成「今天 + 指定時分」的起始時間，作為模擬時鐘
        
        # 鍋爐溫度異常狀態機 (State Machine)
        self.temp_state = "NORMAL" # NORMAL, HEATING, COOLING：目前溫度狀態，預設 NORMAL 正常
        self.current_temp = 100.0  # 目前鍋爐溫度暫存值，預設 100.0°C

    def generate_chip_length(self):
        """
        晶片長度邏輯：
        - 95% 正常品：常態分佈，均值 3.0，標準差 0.025 (使 95% 的資料落在 3 ± 0.05 範圍內)
        - 5% 不良品：均值偏離至 3.12 或 2.88 (超出 ±0.05 容許範圍)
        """
        is_normal = np.random.rand() < 0.95  # 抽 0~1 隨機數，<0.95 為正常品，機率 95%
        if is_normal:
            length = np.random.normal(loc=3.0, scale=0.025)  # 正常品：以均值 3.0、標準差 0.025 常態分佈抽樣
        else:
            # 隨機產生偏高或偏低的缺陷品
            offset = np.random.choice([-0.08, 0.08])  # 隨機選偏移量：-0.08(過短) 或 +0.08(過長)
            length = np.random.normal(loc=3.0 + offset, scale=0.02)  # 不良品：以偏移後均值抽樣，模擬超出規格
        return round(float(length), 4)  # 轉 float 並四捨五入到小數第 4 位回傳

    def generate_boiler_temp(self):
        """
        鍋爐溫度邏輯：
        - 平時：100 ± 1 °C
        - 10% 機率觸發升溫異常：每分鐘升 1°C 至 120°C，再每分鐘降 2°C 回至 100°C (模擬預測性保養需求)
        """
        if self.temp_state == "NORMAL":
            # 正常狀態：每次呼叫有 10% 機率觸發異常
            # 10% 機率觸發異常升溫
            if np.random.rand() < 0.10:
                self.temp_state = "HEATING"  # 切換到 HEATING 升溫狀態
                self.current_temp += 1.0  # 本次先 +1°C
            else:
                self.current_temp = 100.0 + np.random.uniform(-1.0, 1.0)  # 維持正常：在 99~101°C 間均勻隨機擾動
                
        elif self.temp_state == "HEATING":
            # 升溫中：每筆 +1°C，模擬故障逐漸加熱
            self.current_temp += 1.0
            if self.current_temp >= 120.0:  # 到達上限 120°C
                self.current_temp = 120.0  # 鎖定上限，避免超過
                self.temp_state = "COOLING"  # 切換到 COOLING，模擬開始維修降溫
                
        elif self.temp_state == "COOLING":
            # 降溫中：每筆 -2°C，降溫比升溫快
            self.current_temp -= 2.0
            if self.current_temp <= 100.0:  # 回到基準 100°C
                self.current_temp = 100.0  # 鎖定回 100°C
                self.temp_state = "NORMAL"  # 恢復正常狀態

        # 加入微小的測量雜訊
        noise = np.random.uniform(-0.2, 0.2) if self.temp_state != "NORMAL" else 0.0  # 異常期間加 ±0.2 雜訊模擬感測器誤差，正常時不加
        return round(float(self.current_temp + noise), 2)  # 加上雜訊後取小數第 2 位回傳

    def generate_humidity(self):
        """
        廠房濕度邏輯：
        - 99% 的機率為 60%
        - 1% 的機率飆升至 90%
        """
        is_normal = np.random.rand() < 0.99  # 抽 0~1 隨機數，<0.99 為正常，機率 99%
        if is_normal:
            humidity = 60.0 + np.random.uniform(-0.5, 0.5)  # 正常：在 59.5~60.5% 間微幅擾動
        else:
            humidity = 90.0 + np.random.uniform(-1.0, 1.0)  # 異常：飆升到 89~91%，模擬漏水/天氣異常
        return round(float(humidity), 1)  # 取小數第 1 位回傳

    def next_row(self):
        """產出一筆完整資料並推進時間 1 分鐘"""
        timestamp_str = self.current_sim_time.strftime("%Y-%m-%d %H:%M:%S")  # 把目前模擬時間格式化成字串
        chip_length = self.generate_chip_length()  # 呼叫上方方法產生晶片長度
        boiler_temp = self.generate_boiler_temp()  # 呼叫上方方法產生鍋爐溫度 (會更新狀態機)
        humidity = self.generate_humidity()  # 呼叫上方方法產生廠房濕度
        
        row = [self.seq, timestamp_str, chip_length, boiler_temp, humidity]  # 組成一列資料，對應 Excel 標頭順序
        
        # 推進狀態
        self.seq += 1  # 序號 +1，準備下一筆
        self.current_sim_time += datetime.timedelta(minutes=1)  # 模擬時鐘往前推 1 分鐘，模擬每分鐘採樣一次
        
        return row  # 回傳這一筆完整資料列

# 初始化生成器
generator = FactoryDataGenerator(start_hour=8, start_minute=0)  # 建立產生器實例，從 08:00 開始
print("生產數據生成器已準備完成。")  # 提示：初始化成功，後續可呼叫 generator.next_row() 取資料

In [ ]:
# ==========================================
# 3. 主程序：每秒寫入一筆資料至 Excel (按 Ctrl+C 或 Stop 鈕停止)
# ==========================================
# 這一段是無限迴圈：每 1 秒產生 1 筆資料並存進 Excel，模擬即時產線

print("🚀 開始生成資料並寫入 Excel...")  # 提示：程式開始執行
print("💡 每 1 秒會在背景追加 1 筆假裝為『1分鐘』的生產數據。")  # 說明：現實 1 秒 = 模擬 1 分鐘，為了教學加速展示
print("🛑 若要停止生成，請按下 Jupyter 選單上的 Stop 鍵 ⏹️。")  # 說明停止方式，避免無限跑下去

try:  # 用 try 包住迴圈，才能用 Ctrl+C / Stop 鍵中斷而不報錯
    while True:  # 無限迴圈：條件永遠成立，直到手動停止
        # 1. 產生新數據
        new_row = generator.next_row()  # 呼叫產生器，取得一列 [序號, 時間, 長度, 溫度, 濕度]，內部時間會+1分鐘
        
        # 2. 開啟並追加資料至 Excel（使用 openpyxl 快速追加）
        wb = load_workbook(EXCEL_FILE)  # 重新開啟 Excel 檔，確保抓到最新內容
        ws = wb[SHEET_NAME]  # 選取 Live_Data 工作表
        ws.append(new_row)  # 把新資料列附加到工作表最後一列
        wb.save(EXCEL_FILE)  # 存檔，資料才會真正寫入硬碟
        
        # 3. 印出目前產生的數據供教學展示
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] 寫入資料 -> "  # 顯示現在真實時間，證明程式正在跑
              f"序號: {new_row[0]} | 時間: {new_row[1]} | "  # 顯示序號與模擬時間 (new_row[0][1])
              f"長度: {new_row[2]} mm | 鍋爐溫度: {new_row[3]} °C | 濕度: {new_row[4]} %")  # 顯示三個感測器數值
        
        # 4. 等待 1 秒
        time.sleep(1)  # 暫停 1 秒再跑下一圈，避免 CPU 全速運轉，也方便觀察

except KeyboardInterrupt:  # 捕捉手動中斷訊號 (Ctrl+C / Jupyter Stop)
    print("\n🛑 已手動停止數據生成器。")  # 提示：已正常停止，Excel 內的資料會保留